In [30]:
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
import plotly.express as px
from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.compose import TransformedTargetRegressor

In [13]:
RANDOM_STATE = 12345

In [14]:
df = pd.read_csv("../data/diamonds.csv")

In [15]:
X = df.drop('price', axis = 1)

In [16]:
y = df['price']

In [17]:
X = X.drop(columns = 'Unnamed: 0')

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

In [19]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()

In [20]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_23528/4180746908.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


In [21]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

Функция оценки

In [22]:
def evaluate_pipeline(pipeline, X, y, cv):

    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        "mae": -scores["test_mae"].mean(),
        "rmse": -scores["test_rmse"].mean(),
        "r2": scores["test_r2"].mean()
    }

Эксперимент 1
Удаление признаков x, y, z

In [28]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [29]:
X_no_xyz = X_train.drop(columns=['x', 'y', 'z'])

cat_cols = X_no_xyz.select_dtypes(include="object").columns.tolist()
num_cols = X_no_xyz.select_dtypes(exclude="object").columns.tolist()

preprocessor_no_xyz = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

model_no_xyz = Pipeline([
    ("preprocessor", preprocessor_no_xyz),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

results_no_xyz = evaluate_pipeline(
    model_no_xyz,
    X_no_xyz,
    y_train,
    cv
)

results_no_xyz

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_23528/2704451223.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_no_xyz.select_dtypes(include="object").columns.tolist()


{'mae': np.float64(287.70681742232784),
 'rmse': np.float64(559.523697921452),
 'r2': np.float64(0.9803967945195844)}

Вывод эксперимента: не стоит убирать признаки x, y, z, так как они улучшают качество

Эксперимент 2

Логорифмирование таргета

In [31]:
model_log_target = Pipeline([
    ("preprocessor", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

results_log_target = evaluate_pipeline(
    model_log_target,
    X_train,
    y_train,
    cv
)

results_log_target

{'mae': np.float64(278.18608420137616),
 'rmse': np.float64(575.0141275620608),
 'r2': np.float64(0.979285641663972)}

Вывод эксперимента 2: не стоит логорифмировать таргет, так как это ухудшает качество модели.

Эксперимент 3

Создание дополнительных признаков на основе карата

In [32]:
X_poly = X_train.copy()

X_poly["carat_squared"] = X_poly["carat"] ** 2
X_poly["carat_cubed"] = X_poly["carat"] ** 3

cat_cols = X_poly.select_dtypes(include="object").columns.tolist()
num_cols = X_poly.select_dtypes(exclude="object").columns.tolist()

preprocessor_poly = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

model_poly = Pipeline([
    ("preprocessor", preprocessor_poly),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

results_poly = evaluate_pipeline(
    model_poly,
    X_poly,
    y_train,
    cv
)

results_poly

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_23528/495335647.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_poly.select_dtypes(include="object").columns.tolist()


{'mae': np.float64(276.7098575527778),
 'rmse': np.float64(558.5135254519853),
 'r2': np.float64(0.9804741096748038)}

Вывод эксперимента 3: Создание доп признаков не помогло

Обший вывод: ни один из экспериментов не показал метрику выше, чем у базовой моделим